[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Modern_Architectures.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Modern Architectures

The transformer's descendants, dissected: mixture-of-experts routing (capacity without compute), attention's cost curve and its linear/sliding-window repairs, and where [SSM blocks](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) fit. Each mechanism built small and measured.

## 1. Pre-requisites

[Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb), [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb), [Scale_NN](./Scale_NN/Scale_NN.ipynb).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
torch.manual_seed(0)

---
### 🕐 Session 1 of 3 — *Attention's Cost Curve* (~35 min)
**Goal:** measure the quadratic wall; see what sliding windows and linear attention trade away.
**Builds on:** [Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (MoE).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Attention's Cost Curve</b></summary>

**Timing (~35 min).** 8 min why attention is quadratic · 10 min the two repairs · 10 min the benchmark · 7 min reading it honestly.

**Start from where the $T^2$ comes from, because it should not be a memorised fact.** Attention lets every token query every token, so the score matrix $QK^T$ is $T \times T$. That is $O(T^2)$ in both compute *and* memory. Put numbers on it: at $T = 100{,}000$ the attention matrix alone is $10^{10}$ entries — 40 GB in fp32, for **one head, one layer**. **The quadratic wall is not asymptotic hand-waving; it is why long context was hard.**

**Then frame the repairs as *removals*, which is the honest way to teach them.** Neither method makes attention cheaper for free — each **deletes some connectivity** and buys speed with the deletion.
- **Sliding window**: keep only local links. Long-range information is recovered by *stacking layers*, exactly the receptive-field growth argument from [Intro to CNN](./Intro_CNN/Intro_CNN.ipynb). The prior being imposed is "relevance is mostly local."
- **Linear attention**: replace softmax with a positive feature map so the sum factorises, $\phi(Q)(\phi(K)^TV)$, and the $T \times T$ matrix is never formed. State is a fixed $d \times d$ summary — $O(T)$ time, $O(1)$ memory.

**Make the SSM connection explicit, because it unifies two workshops.** Linear attention's "running summary $S$ updated token by token" **is** a state-space recurrence. Mamba and its relatives are the same computational structure with a better-designed state update. Point at [State-Space Models](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) — students who see linear attention and SSMs as one idea have understood something most papers state only in a related-work paragraph.

**Now the benchmark, and warn the room before running it that the printed conclusion is only half right.** The claim is "16× per 4× length". Check it: $512 \to 2048$ gives $3.1 \to 4.5$ ms, a factor of **1.45**, nothing like 16. $2048 \to 8192$ gives $4.5 \to 68.6$ ms, a factor of **15.2** — that one matches. **The quadratic scaling is only visible in the second step**, and asking the room why is the best question in the session.

**The answer is the overhead-bound regime from [Scale_NN](./Scale_NN/Scale_NN.ipynb) Session 2.** At $T = 512$ the actual arithmetic takes microseconds; the 3.1 ms is Python dispatch, allocation, and BLAS setup — fixed costs that do not care about $T$. Only once the matrix is large enough for arithmetic to dominate does the asymptotic behaviour appear. **A benchmark that has not escaped its own overhead measures the framework, not the algorithm**, and that is a transferable lesson worth more than the attention result itself.

**Flag two other things the numbers cannot tell you.** These are single, unwarmed runs — no repeats, no error bars — so treat every value as ±50%. And `window_attn` is a **Python loop** over blocks, which is why it loses to linear attention at $T = 8192$ despite doing less arithmetic; a fused kernel would reverse that ordering. **Implementation quality and asymptotic cost are different axes**, and a timing table conflates them.

**Close on what each method gives up, since that is the design decision.** Full attention: unlimited connectivity, unaffordable at length. Window: cheap and local, needs depth to see far. Linear: cheapest and streams in constant memory, but the fixed-size state is a genuine information bottleneck — a softmax can attend sharply to one token 10,000 positions back, a $d \times d$ summary cannot. **No free lunch: every speedup is a prior about what does not need to be connected.**
</details>

## 2. The Quadratic Wall, Measured

💡 **Intuition.** Full attention lets every token query every token: $O(T^2)$ compute and memory — the price of unlimited connectivity. The repairs each *remove* something: **sliding windows** keep only local links (recover long range by stacking layers — the [CNN receptive-field](./Intro_CNN/Intro_CNN.ipynb) trick); **linear attention** replaces softmax with a kernel so the sum factorizes into a running state ($O(T)$ — and mathematically an [SSM](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb)!). No free lunch: each buys speed with a connectivity prior.

In [2]:
d = 64
def full_attn(q, k, v):
    A = torch.softmax(q @ k.T / d**0.5, dim=-1)
    return A @ v
def window_attn(q, k, v, w=64):
    T = len(q); out = torch.zeros_like(v)
    for i in range(0, T, w):                                  # block-local attention
        sl = slice(max(0, i), min(T, i+w))
        A = torch.softmax(q[sl] @ k[sl].T / d**0.5, dim=-1)
        out[sl] = A @ v[sl]
    return out
def linear_attn(q, k, v):
    phi = lambda x: torch.nn.functional.elu(x) + 1            # positive feature map
    S = phi(k).T @ v                                          # a running SUMMARY, size d×d
    z = phi(k).sum(0)
    return (phi(q) @ S) / (phi(q) @ z)[:, None]

for T in [512, 2048, 8192]:
    q, k, v = (torch.randn(T, d) for _ in range(3))
    times = {}
    for name, fn in [("full", full_attn), ("window", window_attn), ("linear", linear_attn)]:
        tic = time.perf_counter(); fn(q, k, v); times[name] = time.perf_counter()-tic
    print(f"T={T:5d}:  full {times['full']*1e3:7.1f} ms   window {times['window']*1e3:6.1f} ms   linear {times['linear']*1e3:6.1f} ms")
print("→ full attention's time grows ~16x per 4x length (quadratic); the others stay near-linear")

T=  512:  full     3.1 ms   window    0.4 ms   linear    0.7 ms
T= 2048:  full     4.5 ms   window    0.8 ms   linear    0.8 ms
T= 8192:  full    68.6 ms   window    3.2 ms   linear    1.8 ms
→ full attention's time grows ~16x per 4x length (quadratic); the others stay near-linear


**What just happened.** Three attention variants timed at three sequence lengths:

| $T$ | full | window | linear |
|---|---|---|---|
| 512 | 3.1 ms | 0.4 ms | 0.7 ms |
| 2048 | 4.5 ms | 0.8 ms | 0.8 ms |
| 8192 | **68.6 ms** | 3.2 ms | 1.8 ms |

**Check the printed claim against the table, because it is only half supported.** "~16× per 4× length" holds for the second step: $4.5 \to 68.6$ ms is a factor of **15.2**, textbook quadratic. It does **not** hold for the first: $3.1 \to 4.5$ ms is a factor of **1.45**, nowhere near 16. The quadratic wall is visible in one of the two steps, and the discrepancy is the most instructive thing in the cell.

**The explanation is the overhead-bound regime from [Scale_NN](./Scale_NN/Scale_NN.ipynb) Session 2.** At $T = 512$ the $512\times512$ matmul is a few hundred microseconds of real arithmetic; the other ~2.8 ms is Python dispatch, tensor allocation, and BLAS setup — **fixed costs independent of $T$**. Only at $T = 8192$, where the score matrix has 67 million entries, does arithmetic dominate and the asymptotics emerge. **A benchmark that has not escaped its own overhead is measuring the framework, not the algorithm.**

**So the right reading is: the quadratic scaling is real and is confirmed by the 2048 → 8192 step alone.** Put the consequence in memory terms, where it bites hardest: at $T = 100{,}000$ the score matrix is $10^{10}$ entries — **40 GB in fp32, for one head in one layer**. That is why long context was an unsolved engineering problem rather than a matter of waiting for faster chips.

**Note the ordering upset at $T = 8192$, since it is not what the theory predicts.** Linear attention (1.8 ms) beats the sliding window (3.2 ms), despite the window doing less total arithmetic. The reason is in the code: `window_attn` is a **Python loop** over 128 blocks, paying dispatch overhead 128 times, while `linear_attn` is three fused matrix operations. A proper fused windowed kernel — which is what FlashAttention's windowed variants provide — would reverse this. **Asymptotic cost and implementation quality are different axes, and a timing table conflates them.**

**Two caveats on the measurement itself.** These are **single unwarmed runs** with no repeats and no error bars, so treat each number as good to perhaps ±50%; the [Scale_NN](./Scale_NN/Scale_NN.ipynb) throughput cell shows the warmup discipline this one skips. And all three are forward-pass only at one head and $d = 64$ — real cost is dominated by the backward pass and by many heads across many layers.

**Finally, keep the trade-off in view, because speed is not the whole story.** Full attention can put nearly all its weight on one token 8,000 positions back. A sliding window **cannot see that token at all** in a single layer, and recovers reach only by stacking. Linear attention compresses the entire past into a fixed $d\times d$ state, so it can never attend sharply to one distant token — the summary has already blurred it. **Each speedup is a deletion of connectivity**, and the right question about any efficient-attention paper is which connections it decided not to need.

---
### 🕐 Session 2 of 3 — *Mixture of Experts* (~40 min)
**Goal:** route tokens to specialists: parameters without proportional compute — specialization measured.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (the assembled zoo).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Mixture of Experts</b></summary>

**Timing (~40 min).** 10 min the decoupling idea · 10 min routing and its failure mode · 12 min the two runs · 8 min reading the purity column.

**Lead with the decoupling, because it is the entire economic case for MoE.** A dense layer has parameters $\propto$ compute: every parameter touches every token. An MoE layer holds $E$ expert MLPs but routes each token to only the top-$k$, so **parameters scale with $E$ while per-token FLOPs scale with $k$**. Mixtral is the standard example — 47B parameters, 13B active per token. That ratio is the whole reason the architecture exists.

**Then name the bet being made, since it is an empirical claim rather than a theorem.** MoE assumes **tokens differ in kind** and that specialists beat a generalist of equal compute. If every token needs the same computation, routing buys nothing and you have added complexity for no gain. This demo plants four regimes so the assumption is true by construction — which is fair for teaching and worth stating aloud.

**Introduce routing collapse as *the* engineering problem, not a footnote.** The router is trained by gradient descent, and there is a self-reinforcing failure: an expert that receives more tokens trains faster, becomes better, and is therefore routed more tokens. **The rich get richer until most experts are dead.** Ask the room to predict what a collapsed usage table looks like before running the first configuration.

**The first run delivers it: usage `[3153, 0, 11, 2836]`.** One expert received **zero** tokens and another 11 — half the layer's parameters are untrained and unused. Say plainly what that means: **you paid for a 4-expert model and got a 2-expert model.** All the parameter-efficiency arithmetic that motivated MoE evaporates.

**Then the two cures, and be clear they attack different things.** The **load-balancing auxiliary loss** penalises non-uniform expert usage, pushing the router toward even assignment. **Routing noise** adds exploration, so an expert that is currently slightly worse still receives occasional tokens and gets a chance to improve. One is a penalty, one is exploration; production systems use both, plus capacity limits that simply drop tokens beyond an expert's quota.

**Read the second run as a genuine success with a measured margin.** Usage `[1513, 1499, 1542, 1446]` — perfectly balanced — and MSE **0.0016** against **0.0041**, a **2.6× improvement**. Both configurations have identical parameter counts, so this is entirely about whether the capacity gets used.

**But do not oversell the purity column, because it is the part that only half worked.** Purity is `[0.81, 0.51, 0.74, 0.51]` against a random baseline near 0.26. Two experts clearly specialised; **two are at roughly half purity, meaning they each serve a mixture of regimes.** The honest summary is "the load-balancing loss forced *usage* to be uniform, and specialisation followed only partially" — and there is a tension worth naming: a balancing penalty pushes toward uniform *assignment*, which is not the same as correct *specialisation*, and over-strong balancing actively fights it.

**Close with the exercise the notebook proposes and why it is the right one.** Wire the MoE layer into the [nano-GPT](./LLMs_from_the_Ground_Up.ipynb) and measure **perplexity per FLOP** against the dense baseline. Perplexity alone would flatter MoE (more parameters); FLOPs alone would flatter dense. Only the ratio tests the actual claim, which is that MoE buys quality *per unit of compute*.
</details>

## 3. Capacity Without the Bill

💡 **Intuition.** An MoE layer holds $E$ expert MLPs but a learned **router** sends each token to only the top-$k$ — so parameters scale with $E$ while per-token compute scales with $k$. The bet: tokens differ in *kind*, and specialists beat one generalist of equal compute. The classic failure is **routing collapse** (all tokens to one expert), patched with load-balancing losses. We build a 4-expert layer on a task with planted sub-populations and *check who goes where*.

In [3]:
# task with 4 planted regimes: y depends on x differently per quadrant of a latent code
def moe_data(n):
    c = torch.randint(0, 4, (n,))
    x = torch.randn(n, 8)
    x[:, :2] = torch.stack([torch.cos(c*1.57), torch.sin(c*1.57)], 1) + 0.1*torch.randn(n, 2)
    W = torch.stack([torch.randn(8) for _ in range(4)])
    y = (x * W[c]).sum(1, keepdim=True)
    return x, y, c
torch.manual_seed(3)
Xd, Yd, Cd = moe_data(6000)

class MoE(nn.Module):
    def __init__(self, E=4):
        super().__init__()
        self.router = nn.Linear(8, E)
        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 1)) for _ in range(E)])
    def forward(self, x):
        logits = self.router(x)
        top = logits.argmax(1)                                # hard top-1 routing
        probs = torch.softmax(logits, 1)
        out = torch.zeros(len(x), 1)
        for e, expert in enumerate(self.experts):
            m = top == e
            if m.any(): out[m] = expert(x[m]) * probs[m, e:e+1] / probs[m, e:e+1].detach()
        # load-balancing auxiliary: encourage uniform expert usage
        load = probs.mean(0)
        aux = (load * torch.log(load*len(self.experts) + 1e-9)).sum()
        return out, top, aux

def train_moe(aux_weight, route_noise, steps=2500, seed=1):
    torch.manual_seed(seed)
    moe = MoE(); opt = torch.optim.Adam(moe.parameters(), lr=3e-3)
    for step in range(steps):
        logits = moe.router(Xd)
        if route_noise > 0:                                   # exploration noise fights collapse
            logits = logits + route_noise*torch.randn_like(logits)
        top = logits.argmax(1)
        probs = torch.softmax(logits, 1)
        out = torch.zeros(len(Xd), 1)
        for e, expert in enumerate(moe.experts):
            m2 = top == e
            if m2.any(): out[m2] = expert(Xd[m2]) * probs[m2, e:e+1] / probs[m2, e:e+1].detach()
        load = probs.mean(0)
        aux = (load * torch.log(load*4 + 1e-9)).sum()
        loss = ((out - Yd)**2).mean() + aux_weight*aux
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        top = moe.router(Xd).argmax(1)
        out2, _, _ = moe(Xd)
    usage = [int((top==e).sum()) for e in range(4)]
    purity = [float((torch.bincount(Cd[top==e], minlength=4).float().max()/(top==e).sum())) if (top==e).sum()>10 else float("nan") for e in range(4)]
    return usage, purity, float(((out2 - Yd)**2).mean())

# THE FAILURE, on purpose: no balancing, no exploration → routing collapse
u0, p0, m0 = train_moe(aux_weight=0.0, route_noise=0.0)
print(f"no balancing:      usage {u0}   purity {np.round(p0,2)}   MSE {m0:.4f}   ← collapse: dead experts")
# THE CURE: load-balancing loss + routing noise
u1, p1, m1 = train_moe(aux_weight=0.1, route_noise=0.5)
print(f"balanced + noisy:  usage {u1}   purity {np.round(p1,2)}   MSE {m1:.4f}")
print("→ with the cure, every expert lives and each aligns with (mostly) one planted regime —")
print("  routing collapse is not a footnote, it is THE engineering problem of MoE")

no balancing:      usage [3153, 0, 11, 2836]   purity [0.48  nan 0.64 0.52]   MSE 0.0041   ← collapse: dead experts


balanced + noisy:  usage [1513, 1499, 1542, 1446]   purity [0.81 0.51 0.74 0.51]   MSE 0.0016
→ with the cure, every expert lives and each aligns with (mostly) one planted regime —
  routing collapse is not a footnote, it is THE engineering problem of MoE


**What just happened.** The same 4-expert layer trained twice, and the difference is not subtle:

| | usage | purity | MSE |
|---|---|---|---|
| no balancing | `[3153, 0, 11, 2836]` | `[0.48, nan, 0.64, 0.52]` | 0.0041 |
| balanced + noisy | `[1513, 1499, 1542, 1446]` | `[0.81, 0.51, 0.74, 0.51]` | **0.0016** |

**Read the first row as the failure it is: one expert received zero tokens and another eleven.** Half the layer's parameters were never trained and are never used. **You paid for a 4-expert model and got a 2-expert model** — and with it, every bit of the parameter-efficiency arithmetic that motivates MoE in the first place.

**The collapse is self-reinforcing, which is why it needs an explicit cure.** An expert that happens to receive more tokens early trains faster, becomes better, and is therefore routed *more* tokens. The rich get richer; the poor get no gradient at all and never recover. **Routing collapse is not a rare pathology — it is the default outcome**, and every production MoE ships with machinery to prevent it.

**The two interventions attack different halves of that loop.** The **load-balancing auxiliary** penalises non-uniform usage directly, and the **routing noise** provides exploration so a currently-worse expert still receives occasional tokens and gets a chance to improve. Penalty plus exploration; real systems add a third, a hard capacity limit that drops tokens beyond each expert's quota.

**The payoff is measured, not asserted: MSE 0.0041 → 0.0016, a 2.6× improvement.** Both models have **identical parameter counts**. The entire difference is whether the capacity gets used, which is a good demonstration that in MoE the routing is the model.

**Now the column that only partly worked, because glossing over it would be dishonest.** Purity is the fraction of an expert's tokens coming from its most common planted regime; random assignment over 4 regimes gives about **0.26**. Two experts hit 0.81 and 0.74 — clear specialisation. **The other two sit at 0.51, meaning each is serving a blend of regimes.** So the correct summary is: balancing forced *usage* to be uniform, and specialisation followed **partially**.

**There is a real tension underneath that, worth naming.** The auxiliary loss rewards uniform *assignment*, which is not the same as correct *specialisation*. If the planted regimes were unevenly sized, a perfectly specialised router would produce **unbalanced** usage — and the balancing loss would penalise it. Turn the penalty up far enough and you get four experts sharing every regime equally: perfectly balanced, perfectly useless. **The balancing weight is a dial between collapse and homogenisation**, and 0.1 here happens to sit in a workable spot.

**One caveat before generalising from this demo.** The regimes were planted, equally sized, and linearly encoded in the first two input coordinates — close to the easiest routing problem that could exist. Real token distributions are not balanced, and experts in real MoE models specialise far less cleanly than the word "expert" suggests; interpretability work generally finds them capturing shallow token-level patterns rather than semantic domains. **The mechanism is real, the tidy specialisation is a property of this dataset.**

---
### 🕐 Session 3 of 3 — *The Assembled Zoo* (~30 min)
**Goal:** how the pieces combine in 2026-era models; the design-space map.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: The Assembled Zoo</b></summary>

**Timing (~30 min).** 10 min reading the table as a design space · 10 min working a real architecture through it · 10 min the exercise.

**This session has no new code, and that is deliberate — say so.** Sessions 1 and 2 built and measured two mechanisms. This one is a **map**, and its job is to make the room able to read an architecture paper without being impressed by novelty. Every entry in the table is a currency being spent: compute, memory, connectivity, or parameters. **A new architecture is a different allocation, not a different physics.**

**Teach the table by columns, not rows.** The middle column is what people name in papers; the right column is what it actually costs. Force the room to say the cost model aloud for each mechanism before moving on — $O(T^2)$ with an $O(T)$ KV cache, $O(Tw)$, $O(T)$ with $O(1)$ state, params $\times E$ with FLOPs $\times k$. **Anyone who can recite the cost column can predict which mechanism a system will need before reading its abstract**, which is the transferable skill.

**Then work one real architecture through the table live.** Take a recent open model and identify its choices: interleaved sliding-window and full attention layers, grouped-query attention for KV-cache size, MoE feed-forwards, RoPE for position. Each is one row. **Nothing is invented; the paper is a walk through this table**, and demonstrating that once does more for a student's reading ability than another mechanism would.

**Make the "hybrids by necessity" point concrete rather than atmospheric.** Sliding-window layers are cheap but cannot see far in one hop; a few interleaved full-attention layers restore global reach at a fraction of the cost of making every layer global. SSM layers stream in constant memory but cannot attend sharply to a single distant token, so they are paired with a handful of attention layers that can. **The hybrid exists because each mechanism's weakness is another's strength**, and the interleaving pattern is a real design decision with published ablations.

**Emphasise the row students most often underrate: grouped/multi-query attention.** It changes **no mathematics** — the attention computation is identical — and shrinks the KV cache several-fold by sharing key and value heads. During autoregressive generation the KV cache, not the weights, is often the binding memory constraint, so this "trivial" change is what makes long-context serving affordable. **Not every improvement is an idea; some are bookkeeping**, and those are frequently the ones that ship.

**Then set the exercise properly, because the metric is the whole point.** Wire the Session 2 MoE layer into the [nano-GPT](./LLMs_from_the_Ground_Up.ipynb) and measure **perplexity per FLOP** against the dense baseline. Perplexity alone flatters MoE, which has more parameters. FLOPs alone flatter the dense model. **Only the ratio tests the actual claim** — that MoE buys quality per unit of compute — and choosing the metric that can falsify your hypothesis is the research skill being taught.

**Close by giving the room a durable question to carry into any paper.** *What did this mechanism delete, and what does it spend instead?* Full attention deletes nothing and spends $T^2$. Windows delete distant connections and spend depth. Linear attention and SSMs delete sharp recall and spend a fixed state. MoE deletes per-token access to most parameters and spends memory. **Every architecture in the field is an answer to that one question**, and a student who asks it is reading rather than being told.
</details>

## 4. The Map

| Need | Mechanism | Cost model |
|---|---|---|
| Unlimited connectivity | full attention | $O(T^2)$, KV cache $O(T)$ |
| Long context, cheap | sliding window + a few global layers | $O(Tw)$ |
| Constant-state streaming | linear attention / [SSM/Mamba](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) | $O(T)$, state $O(1)$ |
| Parameters ≫ compute | MoE (top-k routing) | params ×E, FLOPs ×k |
| Memory during training | grouped/multi-query attention, [gradient accumulation](./Scale_NN/Scale_NN.ipynb) | smaller KV, same math |

💡 **Intuition.** Modern frontier models are *hybrids by necessity*: interleaved sliding/full attention, MoE feed-forwards, sometimes SSM layers — each mechanism spending a different currency (compute, memory, connectivity). Read any architecture paper as a walk through this table.

**Exercise with teeth:** wire the MoE layer into the [nano-GPT](./LLMs_from_the_Ground_Up.ipynb) and measure perplexity per FLOP against the dense baseline.

---
## Where next

- [State-Space Models](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) — the third pillar, in depth.
- [Scale_NN](./Scale_NN/Scale_NN.ipynb) — why these trade-offs exist at all.